# Phase 8 - End-to-End Pipeline Orchestration

This notebook runs or resumes Phases 1 through 7 through the canonical `run_pipeline.py` entry point.

In [1]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'configs' / 'config.yaml'
print(f'Project root: {project_root}')

Project root: e:\Paper Multiclass-Intrusion-Detection-System


In [2]:
# Cell 2 - Configure a resumable all-scenario pipeline run.
from run_pipeline import configure_logging, run_pipeline

SCENARIO = 'all'
SKIP_PREPROCESSING = True
FORCE_PHASES = set()
DEVICE = 'auto'

configure_logging('INFO')
print(f'Scenario: {SCENARIO}')
print(f'Skip preprocessing: {SKIP_PREPROCESSING}')
print(f'Forced phases: {sorted(FORCE_PHASES) or "none"}')

Scenario: all
Skip preprocessing: True
Forced phases: none


In [3]:
# Cell 3 - Run the canonical Phase 1-7 pipeline or reuse validated artifacts.
pipeline_report = run_pipeline(
    config_path=config_path,
    scenario=SCENARIO,
    skip_preprocessing=SKIP_PREPROCESSING,
    force_phases=FORCE_PHASES,
    device=DEVICE,
)

print(f"Pipeline report: {pipeline_report['report_path']}")

2026-07-17 12:17:13 | INFO | Pipeline started for s1_none, s2_class_weight, s3_upsampling, s4_downsampling. Forced phases: none.
2026-07-17 12:17:13 | INFO | Phase 1/7 started: Dataset inspection (reused).
2026-07-17 12:17:13 | INFO | Phase 1/7 completed in 0.00 seconds.
2026-07-17 12:17:13 | INFO | Phase 2/7 started: Preprocessing (reused).
2026-07-17 12:17:13 | INFO | Phase 2/7 completed in 0.00 seconds.
2026-07-17 12:17:13 | INFO | Phase 3/7 started: Autoencoder training (reused).
2026-07-17 12:17:13 | INFO | Phase 3/7 completed in 0.00 seconds.
2026-07-17 12:17:13 | INFO | Phase 4/7 started: Latent feature extraction (reused).
2026-07-17 12:17:13 | INFO | Phase 4/7 completed in 0.00 seconds.
2026-07-17 12:17:13 | INFO | Phase 5/7 started: Class imbalance handling (validated).
2026-07-17 12:17:23 | INFO | Phase 5/7 completed in 10.50 seconds.
2026-07-17 12:17:23 | INFO | Phase 6/7 started: LightGBM training (validated).
2026-07-17 12:17:25 | INFO | Phase 6/7 completed in 1.27 second

Pipeline report: E:\Paper Multiclass-Intrusion-Detection-System\results\metrics\pipeline_report.json


In [4]:
# Cell 4 - Display execution status and duration for every pipeline phase.
phase_summary = pd.DataFrame(pipeline_report['phases'])
phase_summary

,phase,name,status,duration_seconds
0,1,Dataset inspection,reused,0.000000
1,2,Preprocessing,reused,0.000557
2,3,Autoencoder training,reused,0.000751
3,4,Latent feature extraction,reused,0.000820
4,5,Class imbalance handling,validated,10.502157
5,6,LightGBM training,validated,1.271164
6,7,Evaluation and analysis,executed,4.388042


In [5]:
# Cell 5 - Validate successful completion and the required all-scenario outputs.
assert pipeline_report['success'] is True
assert phase_summary['phase'].tolist() == list(range(1, 8))
assert phase_summary['status'].isin({'executed', 'reused', 'validated'}).all()

summary_path = Path(pipeline_report['evaluation_artifacts']['summary_path'])
analysis_path = Path(pipeline_report['evaluation_artifacts']['analysis_markdown_path'])
assert summary_path.exists()
assert analysis_path.exists()

summary_comparison = pd.read_csv(summary_path)
assert summary_comparison.shape[0] == 4
summary_comparison

,scenario,accuracy,macro_precision,macro_recall,macro_f1,training_seconds,prediction_seconds
0,s1_none,0.843600,0.320398,0.337571,0.314168,104.975497,10.831626
1,s2_class_weight,0.728500,0.410582,0.562000,0.424949,104.912774,10.577099
2,s3_upsampling,0.727819,0.410291,0.557990,0.423284,532.179907,12.209939
3,s4_downsampling,0.615504,0.305047,0.490024,0.292287,1.636508,13.964128


In [6]:
# Cell 6 - Inspect the persisted orchestration metadata for reproducibility.
with open(pipeline_report['report_path'], 'r', encoding='utf-8') as file:
    persisted_report = json.load(file)

{
    'started_at': persisted_report['started_at'],
    'completed_at': persisted_report['completed_at'],
    'total_duration_seconds': persisted_report['total_duration_seconds'],
    'target_scenarios': persisted_report['target_scenarios'],
    'effective_force_phases': persisted_report['effective_force_phases'],
}

{'started_at': '2026-07-17T12:17:13+07:00',
 'completed_at': '2026-07-17T12:17:29+07:00',
 'total_duration_seconds': 16.177398,
 'target_scenarios': ['s1_none',
  's2_class_weight',
  's3_upsampling',
  's4_downsampling'],
 'effective_force_phases': []}